In [0]:
from pyspark.sql.functions import (window,avg,max,min,sum,count,col)

In [0]:
silver_stream_df = (
    spark.readStream
        .format("delta")
        .load(
            "s3a://sebastian-crypto-lakehouse/silver/crypto_transactions/"
        )
)

In [0]:
gold_df = (
    silver_stream_df
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("symbol")
    )
    .agg(
        avg("price").alias("avg_price"),
        max("price").alias("max_price"),
        min("price").alias("min_price"),
        sum("quantity").alias("total_volume"),
        count("*").alias("trade_count")
    )
)

In [0]:
gold_query = (
    gold_df.writeStream
        .format("delta")
        .outputMode("complete")
        .option(
            "checkpointLocation",
            "s3a://sebastian-crypto-lakehouse/checkpoints/gold/market_metrics/"
        )
        .trigger(availableNow=True)
        .start(
            "s3a://sebastian-crypto-lakehouse/gold/market_metrics/"
        )
)

In [0]:
gold_table_df.createOrReplaceTempView("gold_metrics")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM gold_metrics
    """)
)